# gunflows quickstart: train + sample against ToyLLH

`pip install gunflows` (no GUNDAM needed). ToyLLH is a GUNDAM-free demo likelihood: 50 iid Gaussian dims + 10 iid skew-normal dims.

In [ ]:
import torch
from gunflows.dataset import StreamingDataset
from gunflows.utils.build_flow import build_base, build_flow_layers, build_model
from gunflows.losses.importance_losses import kl_symmetric

## 1. Dataset: samples ToyLLH in the background

In [ ]:
import tempfile
data_dir = tempfile.mkdtemp(prefix="gunflows_toyllh_", dir="/tmp")

dataset = StreamingDataset(
    phase_space_dim=list(range(50, 60)),
    with_sampler=True,
    sampler_target="apps.toyllh.likelihood.ToyLLH",
    llh_config="toy",
    gen_batch_size=2000,
    num_samplers=2,
    max_batches=4,
    data_is_asimov=False,
    plot_grid=False,
    data_dir=data_dir,
)
len(dataset)

## 2. Model: small conditional normalizing flow

In [ ]:
base = build_base(dataset.ndim)
tail_bounds = torch.ones(len(dataset.phase_space_dim)) * 6.0
flows = build_flow_layers(
    nflows=4, dim_spline=len(dataset.phase_space_dim), hidden=64, nlayers=1,
    nbins=8, tail_bounds=tail_bounds, n_context=dataset.ndim - len(dataset.phase_space_dim),
)
model = build_model(base, flows, dataset, n_context_flows=2, hidden_dim=64, device="cpu")

## 3. Train

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
for step in range(200):
    if step % 20 == 0:
        dataset.refresh_if_ready(plot_grid=False)
    idx = torch.randint(0, len(dataset), (512,))
    loss = kl_symmetric(model, dataset, idx, stage=0)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 20 == 0:
        print(step, loss.item())

## 4. Sample from the trained model

In [ ]:
with torch.no_grad():
    z_std, _ = model.sample(2000)
samples_phys = (z_std * dataset.std_per_dim + dataset.mean).numpy()
samples_phys[:, dataset.phase_space_dim].mean(axis=0)

In [ ]:
import matplotlib.pyplot as plt
plt.hist(samples_phys[:, 55], bins=40, density=True)
plt.title("Trained NF samples, one skew-normal dim")
plt.show()

## 5. Cleanup

In [ ]:
import shutil
dataset.close()
shutil.rmtree(data_dir, ignore_errors=True)